In [5]:
#!pip -q install -U bertopic  hdbscan  plotly


In [2]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

train_df = pd.read_csv("preprocessed_train_final.csv")


In [3]:
train_df = train_df.groupby('category_fixed', ).sample(n=300, replace=True).reset_index(drop=True)

In [4]:
import json
category_mapping = json.load(open('category_map.json', 'r', encoding='utf-8'))


In [6]:
import re
import pandas as pd

TEXT_COL = "text_for_model"

dfT = train_df[["product_id", TEXT_COL]].copy()
dfT[TEXT_COL] = dfT[TEXT_COL].fillna("").astype(str)

def clean_for_topics(s: pd.Series) -> pd.Series:
    s = s.str.lower()
    # replace punctuation with space
    s = s.str.replace(r"[^\w\sğüşöçıİĞÜŞÖÇ]", " ", regex=True)
    # remove numeric+unit patterns
    s = s.str.replace(r"\b\d+([.,]\d+)?\s*(mm|cm|m|ml|l|lt|gr|g|kg|mg)\b", " ", regex=True)
    s = s.str.replace(r"\b\d+([.,]\d+)?(mm|cm|m|ml|l|lt|gr|g|kg|mg)\b", " ", regex=True)
    # remove packaging tokens
    s = s.str.replace(r"\b\d+\s*(adet|li|lü|lu|lı)\b", " ", regex=True)
    s = s.str.replace(r"\bx\s*\d+\b", " ", regex=True)
    # collapse spaces
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

dfT["topic_text"] = clean_for_topics(dfT[TEXT_COL])

# junk filter (light)
keep = (dfT["topic_text"].str.len() >= 5) & dfT["topic_text"].str.contains(r"[a-zA-ZğüşöçıİĞÜŞÖÇ]", regex=True)
dfT = dfT.loc[keep].reset_index(drop=True)

dfT[["product_id","topic_text"]].head()


,product_id,topic_text
0,214995014,rx3 ax1800 dualband gigabit wi fi6 router
1,2573710,5 port pl gsd 503 10 100 1000 mbps gigabit eth...
2,153372990,usb archer t4u dualband ac1200 867 300mbps
3,46500926,e jl682a 1930s 24port gıgabıt swıtch 4 sfp jl3...
4,165235072,link w8961n 300mbps yüksek hızlı kesintisiz ad...


In [8]:
import numpy as np

SEED = 42
SAMPLE_N = min(500_000, len(dfT))   # tune 200k–500k
dfS = dfT.sample(n=SAMPLE_N, random_state=SEED).reset_index(drop=True)

docs = dfS["topic_text"].tolist()
len(docs), docs[0][:120]


(341966, 'leke çıkarıcı')

In [10]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import umap
import hdbscan

embedding_model_name = "Trendyol/TY-ecomm-embed-multilingual-base-v1.2.0"  # try -small if slow
embedder = SentenceTransformer(embedding_model_name, device="cuda", trust_remote_code=True)

umap_model = umap.UMAP(
    n_neighbors=30,
    n_components=5,
    min_dist=0.1,
    metric="cosine",
    random_state=SEED
)

hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=200,          # tune: higher -> fewer topics, more stable
    min_samples=20,
    metric="euclidean",            # on UMAP space
    cluster_selection_method="eom",
    prediction_data=True
)

topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True
)


In [ ]:
topics, probs = topic_model.fit_transform(docs)
topic_model.get_topic_info().head(10)m  

2026-02-08 16:07:15,046 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 10687/10687 [08:42<00:00, 20.45it/s]
2026-02-08 16:16:01,809 - BERTopic - Embedding - Completed ✓
2026-02-08 16:16:01,809 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-08 16:31:57,890 - BERTopic - Dimensionality - Completed ✓
2026-02-08 16:31:57,905 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-08 18:08:18,494 - BERTopic - Cluster - Completed ✓
2026-02-08 18:08:18,766 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-08 18:08:23,507 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,55698,-1_kazak_tohumu_manto_yağı,"[kazak, tohumu, manto, yağı, lambası, kaban, p...","[yavru kediler için biberon ve süt tozu seti, ..."
1,0,1707,0_bileklik_bilezik_kelepçe_ayar,"[bileklik, bilezik, kelepçe, ayar, altın, 22, ...","[rose beyaz altın kelepçe bilezik, beyaz deri ..."
2,1,1589,1_sabun_sabunu_şampuanı_pişik,"[sabun, sabunu, şampuanı, pişik, baby, sıvı, p...","[bebek sabunu, sabun, sıvı sabun x]"
3,2,1542,2_kumandalı_araba_uzaktan_kamyon,"[kumandalı, araba, uzaktan, kamyon, akülü, kam...","[uzaktan kumandalı spor araba 3 yaş turkuaz, a..."
4,3,1515,3_kulaklık_kulaklığı_mikrofonlu_kulak,"[kulaklık, kulaklığı, mikrofonlu, kulak, bluet...",[lg uyumlu beyaz kablosuz kulaküstü bluetooth ...
5,4,1510,4_bataryası_musluk_duş_lavabo,"[bataryası, musluk, duş, lavabo, musluğu, eviy...",[mutfak eviye bataryası ve arıtma musluğu mutf...
6,5,1409,5_yüzük_tektaş_karat_beştaş,"[yüzük, tektaş, karat, beştaş, pırlanta, yüzüğ...","[çelik taşlı yüzük, mavi taşlı altın yüzük, ka..."
7,6,1370,6_çamurluk_rüzgarlığı_venti_izgarası,"[çamurluk, rüzgarlığı, venti, izgarası, kapı, ...",[volkswagen jetta 2011 2017 uyumlu esnek kauçu...
8,7,1359,7_maske_ffp2_maskesi_meltblown,"[maske, ffp2, maskesi, meltblown, n95, cerrahi...","[ffp2 maske yeşil renk, ffp2 maske beyaz renk,..."
9,8,1339,8_iphone_ekran_kamera_lcd,"[iphone, ekran, kamera, lcd, dokunmatik, galax...",[iphone 12 pro uyumlu kamera metal cam lens ko...


In [14]:
topic_model_5 = topic_model.reduce_topics(docs, nr_topics=5)
topic_model_5.get_topic_info()


2026-02-08 18:18:30,130 - BERTopic - Topic reduction - Reducing number of topics
2026-02-08 18:18:30,378 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-08 18:18:33,071 - BERTopic - Representation - Completed ✓
2026-02-08 18:18:33,103 - BERTopic - Topic reduction - Reduced number of topics from 699 to 5


,Topic,Count,Name,Representation,Representative_Docs
0,-1,55698,-1_seti_ve_kadın_siyah,"[seti, ve, kadın, siyah, beyaz, büyük, set, be...",[büyük beden erkek bordo interlok kumaş mevsim...
1,0,85416,0_uyumlu_siyah_seti_ve,"[uyumlu, siyah, seti, ve, kalem, duvar, set, b...",[volvo xc40 2018 ve sonrası uyumlu oto koltuk ...
2,1,75937,1_kadın_erkek_siyah_beden,"[kadın, erkek, siyah, beden, büyük, çantası, t...",[kadın mavi yaka detaylı büyük beden tensel bl...
3,2,74169,2_takımı_kişilik_beyaz_seti,"[takımı, kişilik, beyaz, seti, ahşap, ve, yata...","[çift kişilik yorgan yastık seti, else siyah b..."
4,3,50746,3_kedi_ve_köpek_saç,"[kedi, ve, köpek, saç, paket, kremi, bakım, te...",[marka saç tarağı mor kategori saç fırçası ve ...


In [16]:
from tqdm.auto import tqdm
import numpy as np

all_docs = dfT["topic_text"].tolist()
N = len(all_docs)

B = 50_000
all_topics = np.empty(N, dtype=np.int32)
all_conf = np.empty(N, dtype=np.float32)

for start in tqdm(range(0, N, B), desc="Assign topics"):
    end = min(start+B, N)
    t, p = topic_model_5.transform(all_docs[start:end])
    all_topics[start:end] = np.array(t, dtype=np.int32)
    # confidence: max prob if available, else 1.0
    if p is None:
        all_conf[start:end] = 1.0
    else:
        all_conf[start:end] = p.max(axis=1).astype(np.float32)

dfT["topic_id"] = all_topics
dfT["topic_conf"] = all_conf
dfT["topic_id"].value_counts().head(20)


Batches: 100%|██████████| 1563/1563 [01:15<00:00, 20.63it/s]
2026-02-08 18:21:35,325 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-02-08 18:22:24,786 - BERTopic - Dimensionality - Completed ✓
2026-02-08 18:22:24,786 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-02-08 18:22:39,046 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-02-08 18:37:50,543 - BERTopic - Probabilities - Completed ✓
2026-02-08 18:37:50,544 - BERTopic - Cluster - Completed ✓
Batches: 100%|██████████| 1563/1563 [01:14<00:00, 20.92it/s]/it]
2026-02-08 18:39:06,395 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-02-08 18:39:24,666 - BERTopic - Dimensionality - Completed ✓
2026-02-08 18:39:24,667 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-02-08 18:39:41,380 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-02-08 18:54:40,952 - 

topic_id
 0    84313
 1    74097
 2    73062
-1    60209
 3    50285
Name: count, dtype: int64

In [17]:
largest_topic = dfT.loc[dfT["topic_id"] != -1, "topic_id"].value_counts().idxmax()
dfT["topic_id_fixed"] = dfT["topic_id"].where(dfT["topic_id"] != -1, largest_topic)

out = (dfT.groupby("topic_id_fixed")["product_id"]
         .apply(list)
         .reset_index(name="product_ids"))

out["topic_keywords"] = out["topic_id_fixed"].map(
    lambda t: ", ".join([w for w,_ in (topic_model_5.get_topic(t) or [])][:10])
)

out


,topic_id_fixed,product_ids,topic_keywords
0,0,"[214995014, 2573710, 153372990, 46500926, 1652...","uyumlu, siyah, seti, ve, kalem, duvar, set, be..."
1,1,"[151523180, 208791519, 189495564, 166883357, 7...","kadın, erkek, siyah, beden, büyük, çantası, ta..."
2,2,"[190190310, 190165651, 120097373, 190219540, 1...","takımı, kişilik, beyaz, seti, ahşap, ve, yatak..."
3,3,"[60021013, 65338141, 62263817, 59793222, 66186...","kedi, ve, köpek, saç, paket, kremi, bakım, tem..."


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
import numpy as np

docs = dfS["topic_text"].tolist()  # sample to fit

tfidf = TfidfVectorizer(min_df=5, max_df=0.6, ngram_range=(1,2), max_features=200_000)
X = tfidf.fit_transform(docs)

nmf = NMF(n_components=5, random_state=SEED, init="nndsvda", max_iter=300)
W = nmf.fit_transform(X)   # doc-topic
H = nmf.components_        # topic-term

vocab = np.array(tfidf.get_feature_names_out())
for k in range(5):
    top = np.argsort(-H[k])[:15]
    print(f"\nTopic {k}:", ", ".join(vocab[top]))



Topic 0: kadın, kadın siyah, beyaz, çantası, taşlı, gümüş, takım, ayakkabı, kadın beyaz, detaylı, altın, lacivert, hamile, deri, kırmızı

Topic 1: seti, ve, bebek, beyaz, çocuk, set, ahşap, kedi, mavi, parça, odası, pembe, desenli, köpek, el

Topic 2: büyük, beden, büyük beden, boy, kadın büyük, büyük boy, eşofman, elbise, lacivert, uzun, hırka, yaka, ceket, etek, tunik

Topic 3: kişilik, takımı, tek, çift, tek kişilik, çift kişilik, yatak, örtüsü, pike, nevresim, yatak örtüsü, nevresim takımı, pike takımı, kişilik yatak, kişilik nevresim

Topic 4: siyah, erkek, kadın siyah, deri, erkek siyah, ayakkabı, hakiki, hakiki deri, unisex, eşofman, siyah erkek, siyah renk, renk, spor, ayakkabısı


In [19]:
def assign_nmf_topics(texts, batch=50_000):
    topic_id = np.empty(len(texts), dtype=np.int32)
    conf = np.empty(len(texts), dtype=np.float32)
    for start in tqdm(range(0, len(texts), batch), desc="NMF assign"):
        end = min(start+batch, len(texts))
        Xb = tfidf.transform(texts[start:end])
        Wb = nmf.transform(Xb)
        topic_id[start:end] = Wb.argmax(axis=1).astype(np.int32)
        conf[start:end] = (Wb.max(axis=1) / (Wb.sum(axis=1)+1e-12)).astype(np.float32)
    return topic_id, conf

all_topic_id, all_conf = assign_nmf_topics(dfT["topic_text"].tolist(), batch=50_000)
dfT["nmf_topic_id"] = all_topic_id
dfT["nmf_conf"] = all_conf


NMF assign: 100%|██████████| 7/7 [00:04<00:00,  1.41it/s]


In [23]:
dfT

,product_id,text_for_model,topic_text,topic_id,topic_conf,topic_id_fixed,nmf_topic_id,nmf_conf
0,214995014,rx3 ax1800 dualband gigabit wi fi6 router,rx3 ax1800 dualband gigabit wi fi6 router,0,1.000000,0,1,0.740768
1,2573710,5 port pl gsd 503 10 100 1000 mbps gigabit eth...,5 port pl gsd 503 10 100 1000 mbps gigabit eth...,0,1.000000,0,1,0.576607
2,153372990,usb archer t4u dualband ac1200 867 300mbps,usb archer t4u dualband ac1200 867 300mbps,0,1.000000,0,4,0.504974
3,46500926,e jl682a 1930s 24g 24port gıgabıt swıtch 4 sfp...,e jl682a 1930s 24port gıgabıt swıtch 4 sfp jl3...,0,1.000000,0,1,0.536630
4,165235072,link w8961n 300mbps yüksek hızlı kesintisiz ad...,link w8961n 300mbps yüksek hızlı kesintisiz ad...,0,1.000000,0,0,0.468810
...,...,...,...,...,...,...,...,...
341961,215876549,elektrikli şömine kumandalı 90x25,elektrikli şömine kumandalı 90x25,0,0.132915,0,1,0.593709
341962,77378590,70 lik standart turbolu şömine haznesi,70 lik standart turbolu şömine haznesi,0,0.001387,0,1,0.637561
341963,65392382,elektrikli şömine 1600w dekoratif şömine beyaz,elektrikli şömine 1600w dekoratif şömine beyaz,0,0.206882,0,1,0.894679
341964,52801888,döküm şömine izgarası 10 lu 49 x 35 x 35 cm,döküm şömine izgarası 49 x,0,0.109132,0,1,0.471998


In [24]:
dfT.nmf_topic_id.value_counts()

nmf_topic_id
1    236812
4     40540
0     27864
3     21074
2     15676
Name: count, dtype: int64